In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
import os

!unzip "/content/drive/MyDrive/Share/Geodata/dataset_yolo_bbox.zip" -d "/content/dataset"



Archive:  /content/drive/MyDrive/Share/Geodata/dataset_yolo_bbox.zip
   creating: /content/dataset/dataset_yolo_bbox/
  inflating: /content/dataset/__MACOSX/._dataset_yolo_bbox  
   creating: /content/dataset/dataset_yolo_bbox/images/
  inflating: /content/dataset/__MACOSX/dataset_yolo_bbox/._images  
   creating: /content/dataset/dataset_yolo_bbox/labels/
  inflating: /content/dataset/__MACOSX/dataset_yolo_bbox/._labels  
  inflating: /content/dataset/dataset_yolo_bbox/metadata.csv  
  inflating: /content/dataset/dataset_yolo_bbox/dataset.yaml  
   creating: /content/dataset/dataset_yolo_bbox/images/train/
  inflating: /content/dataset/__MACOSX/dataset_yolo_bbox/images/._train  
   creating: /content/dataset/dataset_yolo_bbox/images/val/
  inflating: /content/dataset/__MACOSX/dataset_yolo_bbox/images/._val  
   creating: /content/dataset/dataset_yolo_bbox/labels/train/
  inflating: /content/dataset/__MACOSX/dataset_yolo_bbox/labels/._train  
   creating: /content/dataset/dataset_yolo_

In [11]:
from pathlib import Path

root = Path("/content/dataset/dataset_yolo_bbox")
print((root / "images/train").exists())
print((root / "images/val").exists())
print((root / "labels/train").exists())
print((root / "labels/val").exists())
print((root / "dataset.yaml").exists())

yaml_path = Path("/content/dataset/dataset_yolo/dataset.yaml")


True
True
True
True
True


In [12]:
from pathlib import Path

yaml_path = Path("/content/dataset/dataset_yolo_bbox/dataset.yaml")

yaml_path.write_text("""path: /content/dataset/dataset_yolo_bbox

train: images/train
val: images/val

names:
  0: kurgany_tselye
  1: kurgany_povrezhdennye
  2: gorodishcha
  3: fortifikatsii
  4: arkhitektury
""", encoding="utf-8")

print(yaml_path.read_text())

path: /content/dataset/dataset_yolo_bbox

train: images/train
val: images/val

names:
  0: kurgany_tselye
  1: kurgany_povrezhdennye
  2: gorodishcha
  3: fortifikatsii
  4: arkhitektury



In [13]:
df = pd.read_csv("/content/dataset/dataset_yolo_bbox/metadata.csv")
df.head()

,split,region,modality,raster_file,image,label,x,y,tile_size,resize_to,...,tile_p98,tile_p98_minus_p2,class_id,class_name,bbox_x1_px,bbox_y1_px,bbox_x2_px,bbox_y2_px,bbox_area_px,bbox_touches_tile_edge
0,train,004_ДЕМИДОВКА,Li,03_Демидовка_Li_карты_g.tif,dataset_yolo_bbox/images/train/004_ДЕМИДОВКА_L...,dataset_yolo_bbox/labels/train/004_ДЕМИДОВКА_L...,1536,1536,2048,1024,...,212.0,95.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,train,004_ДЕМИДОВКА,Li,03_Демидовка_Li_карты_g.tif,dataset_yolo_bbox/images/train/004_ДЕМИДОВКА_L...,dataset_yolo_bbox/labels/train/004_ДЕМИДОВКА_L...,3072,3072,2048,1024,...,230.0,151.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,train,004_ДЕМИДОВКА,Li,03_Демидовка_Li_карты_g.tif,dataset_yolo_bbox/images/train/004_ДЕМИДОВКА_L...,dataset_yolo_bbox/labels/train/004_ДЕМИДОВКА_L...,4608,3072,2048,1024,...,237.0,127.0,2.0,gorodishcha,1654.0,929.0,2047.0,1473.0,213792.0,True
3,train,004_ДЕМИДОВКА,Li,03_Демидовка_Li_карты_g.tif,dataset_yolo_bbox/images/train/004_ДЕМИДОВКА_L...,dataset_yolo_bbox/labels/train/004_ДЕМИДОВКА_L...,6144,3072,2048,1024,...,235.0,136.0,2.0,gorodishcha,118.0,913.0,986.0,1510.0,518196.0,False
4,train,004_ДЕМИДОВКА,Li,03_Демидовка_Li_карты_g.tif,dataset_yolo_bbox/images/train/004_ДЕМИДОВКА_L...,dataset_yolo_bbox/labels/train/004_ДЕМИДОВКА_L...,7680,3072,2048,1024,...,207.0,51.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df.shape

(7418, 36)

In [8]:
from pathlib import Path
import shutil
import random

import pandas as pd


OLD_DIR = Path("/content/dataset/dataset_yolo_bbox")
NEW_DIR = Path("/content/dataset/dataset_yolo_bbox_v2_kurgans_li_ae")

META_PATH = OLD_DIR / "metadata.csv"

KEEP_MODALITIES = {"Li", "Ae"}
KEEP_OLD_CLASS_IDS = {0, 1}

CLASS_ID_MAP = {
    0: 0,  # kurgany_tselye
    1: 1,  # kurgany_povrezhdennye
}

NAMES = {
    0: "kurgany_tselye",
    1: "kurgany_povrezhdennye",
}

NEGATIVE_RATIO = 0.25
RANDOM_SEED = 42


def make_dirs():
    for split in ["train", "val"]:
        (NEW_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (NEW_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def resolve_old_path(path_from_meta, old_dir, kind, split):
    """
    kind: "images" or "labels"
    """
    p = Path(str(path_from_meta))

    # 1. если путь из metadata реально существует
    if p.exists():
        return p

    # 2. Colab-safe fallback: ищем по имени файла внутри OLD_DIR
    candidate = old_dir / kind / split / p.name
    if candidate.exists():
        return candidate

    return p

def parse_yolo_label(path: Path):
    boxes = []
    if not path.exists():
        return boxes

    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue

        cls_id = int(float(parts[0]))
        coords = list(map(float, parts[1:]))

        if cls_id not in KEEP_OLD_CLASS_IDS:
            continue

        new_cls = CLASS_ID_MAP[cls_id]
        boxes.append((new_cls, *coords))

    return boxes


def valid_box(box):
    _, xc, yc, w, h = box
    return (
        0 <= xc <= 1
        and 0 <= yc <= 1
        and 0 < w <= 1
        and 0 < h <= 1
    )


def write_label(path: Path, boxes):
    lines = [
        f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
        for cls_id, xc, yc, w, h in boxes
    ]
    path.write_text("\n".join(lines), encoding="utf-8")


def write_yaml():
    text = f"""path: {NEW_DIR.resolve()}

train: images/train
val: images/val

names:
  0: kurgany_tselye
  1: kurgany_povrezhdennye
"""
    (NEW_DIR / "dataset.yaml").write_text(text, encoding="utf-8")


def main():
    random.seed(RANDOM_SEED)
    make_dirs()

    meta = pd.read_csv(META_PATH)

    # один ряд на изображение — иначе одно изображение может копироваться много раз
    images = meta.drop_duplicates("image").copy()

    # сначала режем по модальности
    images = images[images["modality"].isin(KEEP_MODALITIES)].copy()

    new_rows = []
    copied_images = 0
    positive_images = 0
    negative_images = 0
    total_boxes = 0
    bad_boxes = 0
    skipped_missing = 0

    for _, row in images.iterrows():
        split = row["split"]

        old_img = resolve_old_path(row["image"], OLD_DIR, "images", split)
        old_lbl = resolve_old_path(row["label"], OLD_DIR, "labels", split)

        if not old_img.exists():
            skipped_missing += 1
            continue

        boxes = parse_yolo_label(old_lbl)
        boxes_ok = []

        for b in boxes:
            if valid_box(b):
                boxes_ok.append(b)
            else:
                bad_boxes += 1

        is_positive = len(boxes_ok) > 0

        # negative оставляем не все, чтобы фон не задавил курганы
        if not is_positive and random.random() > NEGATIVE_RATIO:
            continue

        split = row["split"]
        new_img = NEW_DIR / "images" / split / old_img.name
        new_lbl = NEW_DIR / "labels" / split / old_lbl.name

        shutil.copy2(old_img, new_img)
        write_label(new_lbl, boxes_ok)

        copied_images += 1
        total_boxes += len(boxes_ok)

        if is_positive:
            positive_images += 1
        else:
            negative_images += 1

        base = row.to_dict()
        base["image"] = str(new_img)
        base["label"] = str(new_lbl)
        base["is_positive"] = bool(is_positive)
        base["n_objects"] = len(boxes_ok)

        if boxes_ok:
            for cls_id, xc, yc, w, h in boxes_ok:
                obj = base.copy()
                obj["class_id"] = cls_id
                obj["class_name"] = NAMES[cls_id]
                obj["yolo_xc"] = xc
                obj["yolo_yc"] = yc
                obj["yolo_w"] = w
                obj["yolo_h"] = h
                new_rows.append(obj)
        else:
            obj = base.copy()
            obj["class_id"] = None
            obj["class_name"] = None
            obj["yolo_xc"] = None
            obj["yolo_yc"] = None
            obj["yolo_w"] = None
            obj["yolo_h"] = None
            new_rows.append(obj)

    new_meta = pd.DataFrame(new_rows)
    new_meta.to_csv(NEW_DIR / "metadata.csv", index=False)
    write_yaml()

    print("=" * 80)
    print("DONE: v2 kurgans Li+Ae")
    print("dataset:", NEW_DIR)
    print("images copied:", copied_images)
    print("positive images:", positive_images)
    print("negative images:", negative_images)
    print("total boxes:", total_boxes)
    print("bad boxes skipped:", bad_boxes)
    print("missing images skipped:", skipped_missing)

    if not new_meta.empty:
        print("\nImages by split/modality/positive:")
        print(
            new_meta.drop_duplicates("image")
            .groupby(["split", "modality", "is_positive"])
            .size()
        )

        print("\nBBoxes by class:")
        print(
            new_meta[new_meta["is_positive"]]
            .groupby("class_name")
            .size()
        )

    print("\nyaml:", NEW_DIR / "dataset.yaml")
    print("metadata:", NEW_DIR / "metadata.csv")


if __name__ == "__main__":
    main()

DONE: v2 kurgans Li+Ae
dataset: /content/dataset/dataset_yolo_bbox_v2_kurgans_li_ae
images copied: 635
positive images: 368
negative images: 267
total boxes: 4288
bad boxes skipped: 0
missing images skipped: 0

Images by split/modality/positive:
split  modality  is_positive
train  Ae        False          120
                 True           173
       Li        False          110
                 True           153
val    Ae        False           32
                 True             9
       Li        False            5
                 True            33
dtype: int64

BBoxes by class:
class_name
kurgany_povrezhdennye    3275
kurgany_tselye           1013
dtype: int64

yaml: /content/dataset/dataset_yolo_bbox_v2_kurgans_li_ae/dataset.yaml
metadata: /content/dataset/dataset_yolo_bbox_v2_kurgans_li_ae/metadata.csv


In [9]:
!pip install ultralytics -q
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/dataset/dataset_yolo_bbox_v2_kurgans_li_ae/dataset.yaml",

    imgsz=1024,
    epochs=60,
    batch=8,

    # 🔥 важное для твоего датасета
    patience=20,        # early stopping
    cos_lr=True,        # плавный lr

    # 📦 стабильность
    workers=2,
    cache=True,

    # 🎯 мелкие объекты (очень важно!)
    close_mosaic=10,    # отключить mosaic под конец
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/dataset_yolo_bbox_v2_kurgans_li_ae/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=aut

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dac6c852540>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

After running the above cell, you will be prompted to authorize Google Drive access. Follow the instructions to complete the mounting process. Once mounted, your Google Drive files will be accessible at `/content/drive/My Drive/`.

In [22]:
from pathlib import Path
import shutil
import random

import pandas as pd


OLD_DIR = Path("/content/dataset/dataset_yolo_bbox")
NEW_DIR = Path("/content/dataset/dataset_yolo_bbox_v4_kurgans_li_ae_balanced")

META_PATH = OLD_DIR / "metadata.csv"

KEEP_MODALITIES = {"Li", "Ae"}
KEEP_OLD_CLASS_IDS = {0, 1}

CLASS_ID_MAP = {
    0: 0,  # kurgany_tselye
    1: 1,  # kurgany_povrezhdennye
}

NAMES = {
    0: "kurgany_tselye",
    1: "kurgany_povrezhdennye",
}

RANDOM_SEED = 42
NEGATIVE_RATIO = 0.15
DROP_EDGE_OBJECTS = False
MAX_OBJECTS = 20
MIN_VALID_FRACTION = 0.25
MIN_CONTRAST = 3



def make_dirs():
    for split in ["train", "val"]:
        (NEW_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (NEW_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def resolve_old_path(path_from_meta, old_dir, kind, split):
    """
    kind: "images" or "labels"
    """
    p = Path(str(path_from_meta))

    # 1. если путь из metadata реально существует
    if p.exists():
        return p

    # 2. Colab-safe fallback: ищем по имени файла внутри OLD_DIR
    candidate = old_dir / kind / split / p.name
    if candidate.exists():
        return candidate

    return p

def parse_yolo_label(path: Path):
    boxes = []
    if not path.exists():
        return boxes

    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue

        cls_id = int(float(parts[0]))
        coords = list(map(float, parts[1:]))

        if cls_id not in KEEP_OLD_CLASS_IDS:
            continue

        new_cls = CLASS_ID_MAP[cls_id]
        boxes.append((new_cls, *coords))

    return boxes


def valid_box(box):
    _, xc, yc, w, h = box
    return (
        0 <= xc <= 1
        and 0 <= yc <= 1
        and 0 < w <= 1
        and 0 < h <= 1
    )


def write_label(path: Path, boxes):
    lines = [
        f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
        for cls_id, xc, yc, w, h in boxes
    ]
    path.write_text("\n".join(lines), encoding="utf-8")


def write_yaml():
    text = f"""path: {NEW_DIR.resolve()}

train: images/train
val: images/val

names:
  0: kurgany_tselye
  1: kurgany_povrezhdennye
"""
    (NEW_DIR / "dataset.yaml").write_text(text, encoding="utf-8")


def main():
    random.seed(RANDOM_SEED)
    make_dirs()

    meta = pd.read_csv(META_PATH)

    # один ряд на изображение — иначе одно изображение может копироваться много раз
    images = meta.drop_duplicates("image").copy()

    # сначала режем по модальности
    images = images[images["modality"].isin(KEEP_MODALITIES)].copy()

    new_rows = []
    copied_images = 0
    positive_images = 0
    negative_images = 0
    total_boxes = 0
    bad_boxes = 0
    skipped_missing = 0

    for _, row in images.iterrows():
        split = row["split"]

        old_img = resolve_old_path(row["image"], OLD_DIR, "images", split)
        old_lbl = resolve_old_path(row["label"], OLD_DIR, "labels", split)

        if not old_img.exists():
            skipped_missing += 1
            continue

        same_image = meta[meta["image"] == row["image"]]

        edge_ratio = same_image["bbox_touches_tile_edge"].fillna(False).mean()

        if edge_ratio > 0.8:
            continue

        if row["valid_fraction"] < MIN_VALID_FRACTION:
            continue

        if row["tile_p98_minus_p2"] < MIN_CONTRAST:
            continue

        boxes = parse_yolo_label(old_lbl)

        boxes_ok = []
        for b in boxes:
            if valid_box(b):
                boxes_ok.append(b)
            else:
                bad_boxes += 1

        if len(boxes_ok) > MAX_OBJECTS:
            continue

        is_positive = len(boxes_ok) > 0

        if not is_positive and random.random() > NEGATIVE_RATIO:
            continue

        split = row["split"]
        new_img = NEW_DIR / "images" / split / old_img.name
        new_lbl = NEW_DIR / "labels" / split / old_lbl.name

        shutil.copy2(old_img, new_img)
        write_label(new_lbl, boxes_ok)

        copied_images += 1
        total_boxes += len(boxes_ok)

        if is_positive:
            positive_images += 1
        else:
            negative_images += 1

        base = row.to_dict()
        base["image"] = str(new_img)
        base["label"] = str(new_lbl)
        base["is_positive"] = bool(is_positive)
        base["n_objects"] = len(boxes_ok)

        if boxes_ok:
            if len(boxes_ok) > MAX_OBJECTS:
                continue
            for cls_id, xc, yc, w, h in boxes_ok:
                obj = base.copy()
                obj["class_id"] = cls_id
                obj["class_name"] = NAMES[cls_id]
                obj["yolo_xc"] = xc
                obj["yolo_yc"] = yc
                obj["yolo_w"] = w
                obj["yolo_h"] = h
                new_rows.append(obj)
        else:
            obj = base.copy()
            obj["class_id"] = None
            obj["class_name"] = None
            obj["yolo_xc"] = None
            obj["yolo_yc"] = None
            obj["yolo_w"] = None
            obj["yolo_h"] = None
            new_rows.append(obj)

    new_meta = pd.DataFrame(new_rows)
    new_meta.to_csv(NEW_DIR / "metadata.csv", index=False)
    write_yaml()

    print("=" * 80)
    print("DONE: v4 kurgans Li+Ae")
    print("dataset:", NEW_DIR)
    print("images copied:", copied_images)
    print("positive images:", positive_images)
    print("negative images:", negative_images)
    print("total boxes:", total_boxes)
    print("bad boxes skipped:", bad_boxes)
    print("missing images skipped:", skipped_missing)

    if not new_meta.empty:
        print("\nImages by split/modality/positive:")
        print(
            new_meta.drop_duplicates("image")
            .groupby(["split", "modality", "is_positive"])
            .size()
        )

        print("\nBBoxes by class:")
        print(
            new_meta[new_meta["is_positive"]]
            .groupby("class_name")
            .size()
        )

    print("\nyaml:", NEW_DIR / "dataset.yaml")
    print("metadata:", NEW_DIR / "metadata.csv")


if __name__ == "__main__":
    main()

/tmp/ipykernel_3420/4256983757.py:142: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  edge_ratio = same_image["bbox_touches_tile_edge"].fillna(False).mean()


DONE: v4 kurgans Li+Ae
dataset: /content/dataset/dataset_yolo_bbox_v4_kurgans_li_ae_balanced
images copied: 347
positive images: 226
negative images: 121
total boxes: 772
bad boxes skipped: 0
missing images skipped: 0

Images by split/modality/positive:
split  modality  is_positive
train  Ae        False           64
                 True           104
       Li        False           43
                 True            93
val    Ae        False           11
                 True             7
       Li        False            3
                 True            22
dtype: int64

BBoxes by class:
class_name
kurgany_povrezhdennye    430
kurgany_tselye           342
dtype: int64

yaml: /content/dataset/dataset_yolo_bbox_v4_kurgans_li_ae_balanced/dataset.yaml
metadata: /content/dataset/dataset_yolo_bbox_v4_kurgans_li_ae_balanced/metadata.csv


In [23]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/dataset/dataset_yolo_bbox_v4_kurgans_li_ae_balanced/dataset.yaml",
    imgsz=1024,
    epochs=80,
    batch=8,
    patience=25,
    cos_lr=True,
    workers=2,
    cache=True,
    close_mosaic=15,
    name="kurgans_li_ae_v4_yolov8s_balanced",
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/dataset_yolo_bbox_v4_kurgans_li_ae_balanced/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kurgans_li_ae_v4_yolov8s_balanced, nbs=64, nms=False, opset=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dafce9619d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804